# Run all five baselines (Colab)

Trains DenseNet-121, then Swin-L (Ark+'s no-pretraining control), Swin V2-B, ConvNeXt V2-B and RAD-DINO, one after another in a single Colab session. Each model runs **its own training notebook's cells unchanged** (read from the tagged GitHub copy), so the protocol is exactly the one in the submitted abstract and supplemental PDF: same data and split (fingerprint-checked), same recipe, same code (md5-checked), checkpoints and paper outputs saved to Drive as for Ark+.

**Before Run All**
1. Create the server: *Select Kernel → Colab → New Colab Server → GPU → **G4*** (the same 96 GB GPU as Ark+) → runtime version **2026.07** (current when Ark+ trained; newer runtimes can't build the pinned `albumentations`) → name it.
2. Upload the code bundle (first time only): in the VS Code Explorer, right-click `colab_upload/protocol1-baselines-v1.tar.gz` → **Upload to Colab**. The notebook keeps a copy on Drive for later sessions.
3. **Run All.** Two prompts in the first minute: **Google Drive** sign-in, then your **Kaggle** username and API key. After that it runs unattended.

**While it runs:** keep VS Code open and the Mac awake. Progress is printed below, and `chexpert_protocol_v1/runner_status.json` on Drive records which models have finished. If the session ends early (disconnect, time limit), create a server again and **Run All**: finished runs are skipped and an interrupted run continues from its last epoch.

**When it finishes:** remove the server (*Cmd+Shift+P → Colab: Remove Server*) — an idle G4 still costs ~10 compute units/hour.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# Colab's pre-installed stack (pandas, etc.) is compiled against numpy>=2's actual C ABI --
# this isn't just a pip-metadata preference, running them against numpy<2 crashes for real
# ("numpy.dtype size changed, may indicate binary incompatibility"). We previously tried
# downgrading numpy to dodge an *unrelated* numpy-2.0 incompatibility in imgaug (a transitive
# dep of albumentations) -- that broke pandas instead, and the downgrade didn't even take
# effect this-session anyway, since Colab pre-imports numpy before any of our cells run. So:
# make sure numpy is restored to >=2 here (undoing any earlier accidental downgrade left on
# disk), and fix the imgaug incompatibility directly instead (see section 2, patches 4 & 5).
import sys
!{sys.executable} -m pip install -q --break-system-packages "numpy>=2"
import numpy
print('numpy', numpy.__version__, '@', numpy.__file__)
print('kernel python:', sys.executable)

## 1. Google Drive (first, so both sign-in prompts come in the first minute)

In [ ]:
# Mount Drive first, while you're still here (it asks you to sign in). Checkpoints, results and the runner's status file go there.
from google.colab import drive
drive.mount('/content/drive')


## 2. Dataset (same Kaggle CheXpert-small download as every training notebook)

In [ ]:
import os, json, getpass, sys

kaggle_username = input('Kaggle username: ')
kaggle_key = getpass.getpass('Kaggle API key: ')

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
del kaggle_key

!{sys.executable} -m pip install -q --break-system-packages kaggle
os.makedirs('/content/data', exist_ok=True)

# Using the kaggle package's own Python API instead of any CLI form -- it turns out this
# version of the package has no `__main__.py` (`-m kaggle` fails with "cannot be directly
# executed"), and the bare `kaggle` console script hits the same PATH-mismatch class of
# issue we saw with pip earlier. Calling the API in-process sidesteps all of that.
import kaggle
kaggle.api.authenticate()
# quiet=False -- dataset_download_files defaults to quiet=True, which is why this ran with
# zero output before. With it off, kaggle's own tqdm byte-progress bar (MB downloaded, speed,
# ETA) shows up instead of the download appearing to hang silently for several minutes.
kaggle.api.dataset_download_files('ashery/chexpert', path='/content/data', unzip=True, quiet=False)
print('Download complete.')

In [ ]:
# The train.csv/valid.csv paths are prefixed with 'CheXpert-v1.0-small/', but the
# Kaggle extract puts train/ and valid/ directly under /content/data. Symlink to bridge it.
import os
data_root = '/content/data'
link = os.path.join(data_root, 'CheXpert-v1.0-small')
if not os.path.islink(link) and not os.path.exists(link):
    os.symlink(data_root, link)

print(sorted(os.listdir(data_root)))
import pandas as pd
df = pd.read_csv(os.path.join(data_root, 'train.csv'))
print(df.shape, list(df.columns))

## 3. Dependencies and the tagged training code (md5-checked)

In [ ]:
import os, sys, subprocess

# Same packages and pins as every training notebook (and as Ark+'s run). On Colab runtime 2026.07 -- the one that
# was current when Ark+ trained in August 2026 -- this installs exactly as it did for Ark+. On newer runtimes
# (Python 3.13) albumentations==0.4.3 no longer builds; albumentations, imgaug, pretrainedmodels and einops are never
# imported by the training code, so they are then left out and nothing the training computes changes.
_PKGS = ["timm==0.5.4", "albumentations==0.4.3", "opencv-python-headless", "scikit-image", "scikit-learn",
         "simpleitk", "scipy", "pydicom", "yacs", "einops", "transformers", "pretrainedmodels", "imgaug", "pillow", "tqdm"]
_UNUSED = ["albumentations==0.4.3", "imgaug", "pretrainedmodels", "einops"]
_pip = [sys.executable, "-m", "pip", "install", "-q", "--break-system-packages"]
_r = subprocess.run(_pip + _PKGS, capture_output=True, text=True)
FULL_PACKAGE_SET = _r.returncode == 0
if not FULL_PACKAGE_SET:
    print("Full install failed (expected on Python >= 3.13):", _r.stderr.strip().splitlines()[-1:] or "")
    print("Installing without the packages the training code never imports:", _UNUSED)
    _r2 = subprocess.run(_pip + [p for p in _PKGS if p not in _UNUSED], capture_output=True, text=True)
    assert _r2.returncode == 0, _r2.stderr[-2000:]
print("python", sys.version.split()[0], "| full package set (as Ark+):", FULL_PACKAGE_SET)

# imgaug numpy-2 patch, exactly as in the training notebooks -- only when imgaug is installed.
import importlib.util, os as _os
if importlib.util.find_spec('imgaug') is not None:
    _ia_dir = _os.path.dirname(importlib.util.find_spec('imgaug').origin)
    _ia_params = _os.path.join(_ia_dir, 'parameters.py')
    _s = open(_ia_params).read()
    _before = _s
    _s = _s.replace('np.sctypes["float"]', '[np.float16, np.float32, np.float64]')
    _s = _s.replace('from scipy import misc\n', '')
    if _s != _before:
        open(_ia_params, 'w').write(_s)
        print('imgaug patched:', _ia_params)
    else:
        print('imgaug already patched (or patterns not found), skipping:', _ia_params)
    import imgaug
    print('imgaug import OK, version', imgaug.__version__)

# The training code (dataloader/trainer/engine/models) comes from GitHub at a fixed tag, so every baseline
# runs byte-identical code whichever Google account runs the notebook (the older copy lived on one account's
# Drive and had to be re-synced by hand). The md5 check stops here if any file differs by a single byte from
# the code that was verified (an uninterrupted run is byte-identical to the engine Ark+ trained with).
import subprocess, shutil, hashlib
REPO_URL = "https://github.com/fei7fei/chexpert-baselines.git"
CODE_TAG = "protocol1-baselines-v1"
EXPECTED_CODE_MD5 = {
    "engine.py": "c5b6e179169c35481be1d6e54c9179af",
    "trainer.py": "3b368affdd29b1b03475f82474ebfa6a",
    "models.py": "e0ced0f809680aaebfc97ec3fd6d0e14",
    "dataloader.py": "6aba3a804bb10625a49c9027734a7b76",
    "utils.py": "a7c17deb55d24e826c74b46ad848c441",
}
# The GitHub repo is private, so Colab can't clone it without a login. The tagged code therefore comes as a bundle
# (`git archive` of tag CODE_TAG): uploaded once with VS Code's "Upload to Colab" (it lands in /content) and copied to
# Drive, so later sessions find it by themselves. The md5 check below guarantees it is the verified code either way.
BUNDLE = f"{CODE_TAG}.tar.gz"
_uploaded, _on_drive = f"/content/{BUNDLE}", f"/content/drive/MyDrive/chexpert_protocol_v1/{BUNDLE}"
_finetuning = '/content/chexpert-baselines/Repos/ark_plus_repo/Ark_Plus/Finetuning'
if not os.path.exists(_finetuning):
    if os.path.exists(_uploaded):
        os.makedirs(os.path.dirname(_on_drive), exist_ok=True)
        shutil.copy(_uploaded, _on_drive)  # kept on Drive for future sessions
    if os.path.exists(_on_drive):
        os.makedirs('/content/chexpert-baselines', exist_ok=True)
        subprocess.run(["tar", "-xzf", _on_drive, "-C", "/content/chexpert-baselines"], check=True)
        print("Training code from bundle:", _on_drive)
    else:
        _r = subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", CODE_TAG, REPO_URL, "/content/chexpert-baselines"],
                            capture_output=True, text=True)
        assert _r.returncode == 0, (f"No code bundle and git clone failed ({_r.stderr.strip()}). Right-click {BUNDLE} in the "
                                    "VS Code Explorer -> Upload to Colab, then run this cell again.")
if not os.path.exists('/content/chexpert_code'):
    shutil.copytree(_finetuning, '/content/chexpert_code')
for _fn, _md5 in EXPECTED_CODE_MD5.items():
    _got = hashlib.md5(open(os.path.join('/content/chexpert_code', _fn), 'rb').read()).hexdigest()
    assert _got == _md5, f"{_fn}: md5 {_got} != {_md5} -- not the verified protocol code, stop"
sys.path.insert(0, '/content/chexpert-baselines/analysis')  # paper_outputs.py: paper tables and figures
%cd /content/chexpert_code
print(f'Training code {CODE_TAG} loaded; all {len(EXPECTED_CODE_MD5)} files match the verified md5s.')

## 4. Train the queue

In [ ]:
# ===== Train the five baselines one after another =====
# Each model runs its own notebook's cells (everything after the shared setup above), read from the tagged GitHub
# copy in /content/chexpert-baselines, so the code is exactly the committed notebooks'. Safe to re-run: finished runs
# are skipped (not retrained, not re-tested) and an interrupted run continues from the end of its last epoch.
# If one model fails, the error is recorded and the next model starts.
import json, gc, datetime, traceback
import torch
from IPython import get_ipython

QUEUE = ["DenseNet121", "SwinL", "SwinB", "ConvNeXtV2", "RadDino"]   # quick end-to-end check first, then Ark+'s control
NB_DIR = "/content/chexpert-baselines/Colab_Training_Notebooks"
STATUS_PATH = "/content/drive/MyDrive/chexpert_protocol_v1/runner_status.json"
os.makedirs(os.path.dirname(STATUS_PATH), exist_ok=True)

def _now():
    return datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")

status = {"code_tag": CODE_TAG, "gpu": torch.cuda.get_device_name(0),
          "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
          "python": sys.version.split()[0], "full_package_set": FULL_PACKAGE_SET, "started": _now(), "models": {}}

def _save_status():
    with open(STATUS_PATH, "w") as f:
        json.dump(status, f, indent=1)

def model_cells(name):
    nb = json.load(open(os.path.join(NB_DIR, f"{name}_Colab_Baseline.ipynb")))
    cells = nb["cells"]
    _src = lambda c: c["source"] if isinstance(c["source"], str) else "".join(c["source"])
    k = next(i for i, c in enumerate(cells) if "EXPECTED_CODE_MD5" in _src(c))   # end of the shared setup (run above)
    return [(i, _src(c)) for i, c in enumerate(cells) if i > k and c["cell_type"] == "code"]

_ip = get_ipython()
print(f"GPU: {status['gpu']} ({status['gpu_memory_gb']} GB) | python {status['python']} | code {CODE_TAG}")
if status["gpu_memory_gb"] < 80:
    print("NOTE: under 80 GB -- Swin-L will use 2 x 32 gradient accumulation (same 64-image step); Ark+ used the 96 GB G4.")
for name in QUEUE:
    status["models"][name] = {"state": "running", "started": _now()}
    _save_status()
    print(f"\n\n{'=' * 25} RUNNER: {name} started {_now()} {'=' * 25}", flush=True)
    failed = None
    for i, code in model_cells(name):
        try:
            exec(compile(_ip.transform_cell(code), f"<{name} cell {i}>", "exec"), _ip.user_ns)
        except (Exception, SystemExit) as e:
            traceback.print_exc()
            failed = f"cell {i}: {type(e).__name__}: {e}"
            break
    status["models"][name].update({"state": "failed" if failed else "done", "finished": _now(), "error": failed})
    _save_status()
    print(f"\n{'=' * 25} RUNNER: {name} {'FAILED -- ' + failed if failed else 'finished'} {_now()} {'=' * 25}", flush=True)
    gc.collect()
    torch.cuda.empty_cache()
    _ip.run_line_magic("cd", "/content/chexpert_code")
status["finished"] = _now()
_save_status()
print("\nRUNNER: queue complete --", {k: v["state"] for k, v in status["models"].items()})
print("Remove this Colab server now so it stops charging: Cmd+Shift+P -> Colab: Remove Server.")
